In [12]:
import json
import os
import threading
import time
from datetime import datetime, timezone

import paho.mqtt.client as mqtt

MQTT_HOST = os.environ.get("MQTT_HOST", "localhost")
MQTT_PORT = int(os.environ.get("MQTT_PORT", "1883"))
ROBOT_ID = "robot-01"
ROOT = f"iiot/robots/{ROBOT_ID}"
COMMAND_TOPIC = f"{ROOT}/command"
STATUS_TOPIC = f"{ROOT}/status"

DRY_RUN = True  # Set False only on the robot after hardware and safety checks
MQTT_ENABLED = os.environ.get("MQTT_ENABLED", "0").strip().lower() in {"1", "true", "yes", "on"}
OBSTACLE_LIMIT_CM = 25.0
REMOTE_COMMAND_TIMEOUT_S = 2.0
STATUS_PERIOD_S = 1.0
MAX_SPEED = 0.45  # gpiozero speed fraction (0..1)

state_lock = threading.Lock()
state = {"mode": "autonomous", "command": "stop", "distance_cm": None,
         "obstacle": False, "mqtt_connected": False, "last_command_at": 0.0,
         "last_error": None, "running": True}
robot = None
distance_sensor = None
controller_thread = None


In [13]:
if not DRY_RUN and not MQTT_ENABLED:
    raise RuntimeError("Set MQTT_ENABLED=1 before using hardware mode.")

if DRY_RUN:
    simulated_distance_cm = 100.0
    print("Dry-run mode enabled: no GPIO or motors will be controlled.")
else:
    from gpiozero import Robot, DistanceSensor
    robot = Robot(left=(17, 18), right=(22, 27))
    distance_sensor = DistanceSensor(echo=6, trigger=5, max_distance=2.0, threshold_distance=OBSTACLE_LIMIT_CM / 100)
    print("Hardware initialized. Verify emergency stop and motor direction before driving.")

def get_distance_cm():
    if DRY_RUN:
        return simulated_distance_cm
    return distance_sensor.distance * 100.0

def motor_action(command):
    """Apply one supported motor command."""
    if DRY_RUN:
        return
    if command == "forward":
        robot.forward(MAX_SPEED)
    elif command == "backward":
        robot.backward(MAX_SPEED)
    elif command == "left":
        robot.left(MAX_SPEED)
    elif command == "right":
        robot.right(MAX_SPEED)
    else:
        robot.stop()


Dry-run mode enabled: no GPIO or motors will be controlled.


In [14]:
ALLOWED_COMMANDS = {"forward", "backward", "left", "right", "stop"}

def set_command(mode, command):
    with state_lock:
        state["mode"] = mode
        state["command"] = command
        state["last_command_at"] = time.monotonic()

def on_connect(client, userdata, flags, reason_code, properties):
    connected = (reason_code == 0)
    with state_lock:
        state["mqtt_connected"] = connected
        if not connected:
            state["last_error"] = f"MQTT connect failed: {reason_code}"
    if connected:
        client.subscribe(COMMAND_TOPIC, qos=1)
        publish_status(client)

def on_disconnect(client, userdata, disconnect_flags, reason_code, properties):
    with state_lock:
        state["mqtt_connected"] = False
    motor_action("stop")  # Fail safe when broker connection is lost

def on_message(client, userdata, msg):
    try:
        payload = json.loads(msg.payload.decode("utf-8"))
        mode = payload.get("mode", "remote")
        command = payload.get("command", "stop")
        if mode == "autonomous":
            set_command("autonomous", "stop")
            return
        if mode != "remote" or command not in ALLOWED_COMMANDS:
            raise ValueError("mode/command not supported")
        # Obstacle interlock also applies to remote forward commands.
        distance = get_distance_cm()
        if command == "forward" and distance < OBSTACLE_LIMIT_CM:
            command = "stop"
        set_command("remote", command)
    except (UnicodeDecodeError, json.JSONDecodeError, AttributeError, ValueError) as exc:
        with state_lock:
            state["last_error"] = f"Rejected command: {exc}"



In [15]:
def publish_status(client):
    with state_lock:
        snapshot = dict(state)
    snapshot.pop("last_command_at", None)
    snapshot["timestamp"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
    snapshot["robot_id"] = ROBOT_ID
    client.publish(STATUS_TOPIC, json.dumps(snapshot), qos=1, retain=True)

def control_loop(client):
    last_status = 0.0
    while state["running"]:
        try:
            distance = get_distance_cm()
            with state_lock:
                state["distance_cm"] = round(distance, 1)
                state["obstacle"] = distance < OBSTACLE_LIMIT_CM
                connected = state["mqtt_connected"]
                mode = state["mode"]
                command = state["command"]
                last_cmd = state["last_command_at"]

            if not connected and not DRY_RUN:
                motor_action("stop")
            elif mode == "remote":
                if time.monotonic() - last_cmd > REMOTE_COMMAND_TIMEOUT_S:
                    set_command("remote", "stop")
                    motor_action("stop")
                elif command == "forward" and distance < OBSTACLE_LIMIT_CM:
                    motor_action("stop")
                    with state_lock:
                        state["command"] = "stop"
                else:
                    motor_action(command)
            else:  # autonomous
                if distance < OBSTACLE_LIMIT_CM:
                    motor_action("right")
                    with state_lock:
                        state["command"] = "right"
                else:
                    motor_action("forward")
                    with state_lock:
                        state["command"] = "forward"

            if client is not None and state["mqtt_connected"] and time.monotonic() - last_status >= STATUS_PERIOD_S:
                publish_status(client)
                last_status = time.monotonic()
        except Exception as exc:
            with state_lock:
                state["last_error"] = str(exc)
            motor_action("stop")
        time.sleep(0.1)

mqtt_client = None
if MQTT_ENABLED:
    mqtt_client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id=ROBOT_ID)
    mqtt_client.on_connect = on_connect
    mqtt_client.on_disconnect = on_disconnect
    mqtt_client.on_message = on_message
    try:
        mqtt_client.connect(MQTT_HOST, MQTT_PORT, keepalive=20)
    except OSError as exc:
        motor_action("stop")
        if robot is not None:
            robot.close()
        if distance_sensor is not None:
            distance_sensor.close()
        raise RuntimeError(f"Cannot connect to MQTT broker at {MQTT_HOST}:{MQTT_PORT}. Start the broker or check MQTT_HOST/MQTT_PORT.") from exc
    mqtt_client.loop_start()
else:
    print("MQTT disabled; controller running offline.")
controller_thread = threading.Thread(target=control_loop, args=(mqtt_client,), daemon=True)
controller_thread.start()
print(f"Robot controller started. Command topic: {COMMAND_TOPIC}; status topic: {STATUS_TOPIC}")

MQTT disabled; controller running offline.
Robot controller started. Command topic: iiot/robots/robot-01/command; status topic: iiot/robots/robot-01/status


In [16]:
if DRY_RUN:
    simulated_distance_cm = 100.0
    print("Clear path: 100 cm")
    time.sleep(2)
    simulated_distance_cm = 15.0
    print("Obstacle introduced: 15 cm (forward motion will be blocked)")
    time.sleep(2)
    simulated_distance_cm = 100.0
    print("Path clear again: 100 cm")
else:
    print("Simulation skipped because hardware mode is enabled.")

Clear path: 100 cm
Obstacle introduced: 15 cm (forward motion will be blocked)
Path clear again: 100 cm


In [17]:
with state_lock:
    state["running"] = False
motor_action("stop")
if controller_thread is not None:
    controller_thread.join(timeout=3)
if mqtt_client is not None:
    mqtt_client.loop_stop()
    mqtt_client.disconnect()
if robot is not None:
    robot.close()
if distance_sensor is not None:
    distance_sensor.close()
print("Robot stopped and resources released.")

Robot stopped and resources released.
